# Listening Past the Noise: frozen 0.834 ensemble

This notebook runs the previous frozen model. MAP@3 is 0.833871 on five-fold checked-label CV with seed 131; this is not accuracy or a competition test score. Earlier reserved holdout MAP@3 was 0.806452. See README.md for the distinction between evaluations.

Install requirements-tested.txt in this notebook environment and extract the competition data into project4/. Keep this notebook next to the Python scripts.

In [ ]:
from pathlib import Path
import sys,subprocess,json
import pandas as pd
BASE=Path.cwd()
assert (BASE/"improve.py").exists(), "Open this notebook from the repository folder"
JOBS=2
def run(script,*args):
    subprocess.run([sys.executable,"-u",str(BASE/script),*map(str,args)],cwd=BASE,check=True)

In [ ]:
train=pd.read_csv(BASE/"project4/train.csv")
test=pd.read_csv(BASE/"project4/test.csv")
print("Training:",len(train),"Test:",len(test),"Checked:",train.checked.sum())
display(train.groupby("label").agg(total=("id","size"),checked=("checked","sum")))

## Cache deterministic audio features and GMM Fisher vectors

These stages resume from cached results and can take several minutes. No external or pretrained model is used.

In [ ]:
run("improve.py","--stage","features","--jobs",JOBS)
run("improve.py","--stage","fisher","--jobs",JOBS)

## Reproduce the five-fold score

Each fold excludes its checked validation labels from model fitting and propagation seeds. Feature-only transductive steps are described in README.md.

In [ ]:
run("evaluate_frozen.py","--seed",131,"--jobs",JOBS)
print(json.loads((BASE/"validation_0834.json").read_text()))

## Train using all training data and validate the submission

The checked-in selection.json specifies the frozen three-component ensemble. This does not run or select the later round-two model.

In [ ]:
run("deliver.py","--jobs",JOBS)
submission=pd.read_csv(BASE/"submission_improved.csv",dtype=str)
assert submission.id.tolist()==test.id.tolist()
assert submission.id.is_unique
assert all(len(v.split())==len(set(v.split()))==3 and set(v.split())<=set(train.label) for v in submission.label)
display(submission.head())
print("Validated rows:",len(submission))

In [ ]:
run("test_pipeline.py")